# chess-vi — SFT trên Colab

Notebook này **chỉ gọi script** `chessvi.train.sft`. Không copy-paste logic vào cell:
logic nằm trong repo để test được và để Colab với local chạy đúng một thứ.

Runtime cần: **A100 / L4 / T4 GPU**. Colab hay ngắt giữa chừng nên script bật
`hub_strategy="checkpoint"` — cell cuối resume lại từ checkpoint.

Thứ tự chạy: `Mount` → `Cài đặt` → `Token` → `GPU` → `C1 + dịch (T5)` →
`Validate (T6)` → `Smoke test` → `Train`.
Đứt kết nối thì chạy lại 4 cell đầu rồi nhảy thẳng xuống cell **Resume**.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/chessvi"
!mkdir -p {DRIVE_ROOT}/outputs

## 2. Lấy repo và cài đặt

Sửa `REPO_URL` thành remote của bạn. Nếu đã clone repo vào Drive thì chỉ cần `cd`.

In [ ]:
REPO_URL = "https://github.com/trantrien1/ChessVi.git"
REPO_DIR = "/content/ChessVi"

import os

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull --ff-only || true

# Bỏ -q: -q nuốt mất chi tiết xung đột dependency, chỉ còn trơ
# "ResolutionImpossible" không biết package nào đá nhau.
!pip install -e ".[train,data]" 2>&1 | tail -30

# pip thất bại KHÔNG làm notebook dừng lại — không chốt ở đây thì mọi cell sau
# sẽ chết vì ModuleNotFoundError và che mất nguyên nhân thật.
!python -c "import chessvi; print('chessvi OK:', chessvi.__file__)" || echo ">>> CÀI ĐẶT HỎNG — DỪNG LẠI, ĐỪNG CHẠY CELL SAU"

## 3. Token Hugging Face

Lưu token trong **Colab Secrets** (biểu tượng chìa khoá bên trái, tên
`HF_TOKEN`). Không bao giờ dán token thẳng vào cell — notebook sẽ bị commit
kèm token.

**Đổi sang token khác:** sửa giá trị trong Colab Secrets rồi chạy lại cell
này. Secrets là nguồn chuẩn, được đọc trước biến môi trường, nên token cũ
đang nằm trong môi trường sẽ bị ghi đè. (Bản cũ của cell này bỏ qua Secrets
khi môi trường đã có token, nên chạy lại không có tác dụng gì.)

Cell in ra **account và quyền** của token. Muốn push lên Hub thì quyền phải
là `write`, và tên account đó chính là namespace repo — không phải tên
GitHub của bạn.

In [ ]:
import os

# Colab Secrets chỉ đọc được khi chạy từ giao diện web colab.research.google.com:
# userdata.get() hỏi ngược về frontend trong trình duyệt. Chạy từ VS Code hay một
# frontend khác thì không ai trả lời -> TimeoutException. Bắt rộng vì mỗi trường
# hợp hỏng một kiểu (ImportError, TimeoutException, SecretNotFoundError).
def _from_colab_secrets() -> str:
    try:
        from google.colab import userdata

        return userdata.get("HF_TOKEN") or ""
    except Exception as error:
        print(f"Không lấy được Colab Secret ({type(error).__name__}).")
        return ""


# Secrets đọc TRƯỚC biến môi trường: đó là thứ bạn sửa được, nên nó phải thắng.
# Đọc sau thì token cũ trong môi trường sẽ khoá chặt, chạy lại cell vô ích.
token = _from_colab_secrets() or os.environ.get("HF_TOKEN", "")
if not token:
    import getpass

    # getpass không in token ra output, nên notebook commit lên không lộ.
    token = getpass.getpass("HF_TOKEN (Enter để bỏ qua): ")
os.environ["HF_TOKEN"] = token

# Token CHỈ cần khi push lên Hub. Cell validate và smoke test không cần.
if not token:
    print("CHƯA CÓ HF_TOKEN — validate và smoke test vẫn chạy, cell 8 sẽ không push được.")
else:
    from huggingface_hub import HfApi

    info = HfApi(token=token).whoami()
    role = info.get("auth", {}).get("accessToken", {}).get("role", "fine-grained")
    print(f"Account: {info['name']}  |  quyền: {role}")
    if role == "read":
        print(">>> TOKEN CHỈ ĐỌC — cell 8 sẽ 403. Tạo token Write rồi chạy lại cell này.")

## 4. Kiểm tra GPU

In [ ]:
!nvidia-smi

import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

## 5. Lấy và dịch dữ liệu C1 (T5)

`chessvi.data.c1` chuẩn hoá `UofTCSSLab/C1-data` về schema pipeline hiểu: tách
FEN ra khỏi câu văn thành cột riêng, rút nước sau `FINAL_ANSWER:` thành `label`.
Bỏ bước này thì T6 loại 100% mẫu với lý do `missing_fen`.

Backend `llm` mặc định **`Qwen/Qwen3-30B-A3B`**, cần **A100 80GB**. Chỉ có 40GB
thì thêm `--model Qwen/Qwen3-14B`. Vì sao không phải 32B dù nó to hơn: 32B ngốn
65,6GB weights, chỉ còn 3,9GB KV cache nên vLLM chạy được rất ít request song
song — chậm hơn cả 14B. 30B-A3B là MoE, mỗi token chỉ kích hoạt 3,3B.

`--batch-size 256` là đòn bẩy thông lượng lớn nhất, **đừng bỏ flag này**: mặc
định 16 khiến vLLM chỉ thấy 16 request cùng lúc trong khi KV cache đủ chỗ cho
hơn 26x. Đo thật ở batch 16 là 534 tok/s, tức khoảng 9 tiếng cho cả tập.

Sau mỗi lượt sinh có một **cổng chất lượng**: bản dịch còn nguyên tiếng Anh,
rụng mất một đoạn, lệch ký hiệu `<M0>`, hoặc đổi màu quân (Vua Trắng thành Vua
Đen) đều bị nhắc dịch lại một lượt. Hỏng tiếp thì giữ bản gốc tiếng Anh để T6
loại và đếm được, chứ không để bản dịch sai lọt vào tập train. Cuối lần chạy
script in dòng `Cổng chất lượng: ...` — đọc dòng đó trước khi sang T6.

Cell dưới chạy **toàn bộ 39.601 mẫu**, mất vài tiếng. Mọi thứ ghi thẳng vào
Drive nên sống sót qua việc runtime bị xoá:

| Đường dẫn trên Drive | Nội dung |
|---|---|
| `data/raw/c1_sft.jsonl` | C1-data đã chuẩn hoá |
| `data/translated/sft/part-*.parquet` | bản dịch, ghi mỗi 500 mẫu |
| `data/translated/sft/_state.json` | checkpoint cho `--resume` |

Colab ngắt giữa chừng thì chạy lại **đúng cell đó** — `--resume` đọc
`_state.json` và đi tiếp từ chỗ dừng, không dịch lại gì.

Sau khi đổi model hoặc sửa bảng thuật ngữ, muốn soi tay 20 cặp before/after
trước thì thêm `--limit 20 --dry-run` vào lệnh dịch; nó không ghi file.

In [ ]:
import os
import time

RAW = f"{DRIVE_ROOT}/data/raw/c1_sft.jsonl"
TRANSLATED_DIR = f"{DRIVE_ROOT}/data/translated"
TRANSLATED = f"{TRANSLATED_DIR}/sft"

# vLLM nằm trong extra [rl]; cell 2 chỉ cài [train,data].
!pip install -e ".[rl]" 2>&1 | tail -5
!python -c "import vllm" || echo ">>> vLLM HỎNG — DỪNG LẠI, ĐỪNG CHẠY TIẾP"

# Chuẩn hoá đủ 39.601 mẫu. Bỏ qua nếu đã có: --resume bỏ mẫu theo THỨ TỰ DÒNG,
# nên file này phải y hệt nhau giữa các lần chạy.
if os.path.exists(RAW):
    print(f"Đã có {RAW} — bỏ qua bước chuẩn hoá")
else:
    !python -m chessvi.data.c1 --split sft --out {RAW}
!wc -l {RAW}

# Chạy thật, vài tiếng. Ghi thẳng vào Drive, checkpoint mỗi 500 mẫu.
start = time.time()
!python -m chessvi.data.translate --input-jsonl {RAW} --split sft --backend llm --batch-size 256 --out-dir {TRANSLATED_DIR} --resume
print(f"Dịch xong sau {(time.time() - start) / 3600:.2f} giờ")

# Xác nhận dữ liệu nằm trên Drive chứ không phải /content (mất khi runtime chết).
!ls -la {TRANSLATED}
!cat {TRANSLATED}/_state.json

## 6. Validate dữ liệu đã dịch (T6)

`--out-clean` ghi tập đã pass ra `data/validated/sft` — đây chính là đầu vào
của SFT. Không có bước này thì không có file nào chứa "dữ liệu đã validate".

Kỳ vọng loại 5–15%. Trên 25% script sẽ in cảnh báo to: pipeline dịch có vấn đề,
quay lại T5 chứ đừng train.

**Chốt chặn thủ công:** trước khi sang cell tiếp theo, tự đọc tay 200 mẫu ngẫu
nhiên đã pass. Không ai thay bạn làm bước này được.

In [ ]:
TRANSLATED = f"{DRIVE_ROOT}/data/translated/sft"
DATA = f"{DRIVE_ROOT}/data/validated/sft"

!python -m chessvi.data.validate \
    --input {TRANSLATED} \
    --rejected {DRIVE_ROOT}/data/rejected.jsonl \
    --out-clean {DATA}

!ls -la {DATA}

## 7. Smoke test

5 step với model 0.6B. Cell này phải chạy xong không lỗi **trước khi** tốn CU
cho lần train thật.

In [ ]:
!python -m chessvi.train.sft \
    --data {DATA} \
    --base-model Qwen/Qwen3-0.6B \
    --limit 50 --max-steps 5 \
    --output-dir /content/outputs/smoke \
    --no-push

## 8. Train thật — bản 4B (chạy được trên máy local)

Repo Hub được tạo ở chế độ **private**, dưới đúng account của token — tên
account lấy bằng `whoami()` chứ không hardcode. Token thuộc account A mà code
tạo repo dưới account B thì Hub trả `403 Forbidden: You don't have the rights
to create a model under the namespace ...`.

`create_repo` ở đầu cell là chốt chặn 10 giây: `hub_strategy="checkpoint"` đẩy
lên Hub mỗi 200 step, nên token chỉ có quyền đọc sẽ làm cả lần chạy chết ở
phút thứ 30 chứ không phải phút đầu. Token phải là loại **Write**.

`--batch-size 8 --grad-accum 2` giữ batch hiệu dụng 16 như cũ nhưng chạy 8
chuỗi song song thay vì 8 lượt nối tiếp — tham số `1 x 16` là cho GPU 4GB,
phí A100. OOM thì lùi về `4 x 4`.

**1 epoch trước đã.** 39.355 mẫu / 16 = khoảng **2.460 step**. Hai epoch gấp
đôi thời gian lẫn CU, mà LoRA r=32 trên tập hẹp thì epoch thứ hai dễ thành
học thuộc hơn là học thêm. Cổng accuracy sau T8 thấp thì đổi thành
`--epochs 2 --resume`, nó học tiếp chứ không train lại từ đầu.

Xem `it/s` trên thanh tiến trình sau khoảng 50 step rồi nhân với 2.460 để
biết tổng thời gian, trước khi bỏ đi ngủ.

In [ ]:
import os

from huggingface_hub import HfApi, create_repo

# Namespace phải là account của chính token, không phải tên GitHub của bạn.
HF_USER = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
HUB_MODEL_ID = f"{HF_USER}/chessvi-4b-sft"
OUTPUT_DIR = f"{DRIVE_ROOT}/outputs/sft"

# Nổ ngay bây giờ nếu token chỉ có quyền đọc, thay vì ở step 200.
create_repo(HUB_MODEL_ID, private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
print("Repo Hub sẵn sàng:", HUB_MODEL_ID)

!python -m chessvi.train.sft \
    --data {DATA} \
    --base-model Qwen/Qwen3-4B \
    --output-dir {OUTPUT_DIR} \
    --hub-model-id {HUB_MODEL_ID} \
    --epochs 1 --batch-size 8 --grad-accum 2 --save-steps 200

## 8b. Train thật — bản 30B (tuỳ chọn, chỉ chạy được trên cloud)

Cùng dữ liệu, cùng siêu tham số, khác base model và khác thư mục ra — nên chạy
cả hai cell là có hai bộ weight độc lập, sau thích dùng cái nào thì dùng.

**Bản này không chạy được trên RTX 3050 Ti.** Q4 của Qwen3-30B-A3B nặng
~18,6GB, card 4GB không tải nổi; bản 4B ở Q4 là ~2,5GB nên vừa. Muốn dùng 30B
thì phải phục vụ từ cloud.

Script tự nhận base model là MoE và cho LoRA **chỉ bám attention**. Không có
chốt đó thì `all-linear` bám luôn vào 128 expert x 3 phép chiếu x 48 lớp =
18.432 linear, ra ~1,7 tỷ tham số LoRA mà router chỉ kích hoạt 8/128 expert
mỗi token — phần lớn adapter không nhận gradient. Bám attention ra ~27M tham
số huấn luyện.

Batch để 4: MoE giữ toàn bộ expert trong VRAM nên chỗ cho activation ít hơn
bản dense. Chạy trót lọt vài trăm step rồi thì nâng lên 8 cũng được.

Chạy cell 4B trước và đọc thời gian thật của nó. Bản 30B nặng hơn nhiều, nếu
4B đã ngốn quá nửa số CU còn lại thì đừng khởi động bản này.

In [ ]:
import os

from huggingface_hub import HfApi, create_repo

HF_USER = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
HUB_MODEL_ID_30B = f"{HF_USER}/chessvi-30b-sft"
OUTPUT_DIR_30B = f"{DRIVE_ROOT}/outputs/sft-30b"

create_repo(HUB_MODEL_ID_30B, private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
print("Repo Hub sẵn sàng:", HUB_MODEL_ID_30B)

!python -m chessvi.train.sft \
    --data {DATA} \
    --base-model Qwen/Qwen3-30B-A3B \
    --output-dir {OUTPUT_DIR_30B} \
    --hub-model-id {HUB_MODEL_ID_30B} \
    --epochs 1 --batch-size 4 --grad-accum 4 --save-steps 200

## 9. Resume sau khi Colab ngắt

Chạy lại cell 1–4 rồi chạy thẳng cell này — nó **tự định nghĩa lại** `DATA`,
`OUTPUT_DIR`, `HUB_MODEL_ID` nên không cần chạy lại cell 8.

`--resume` đọc checkpoint mới nhất trong `--output-dir`; output nằm trên Drive
nên checkpoint vẫn còn nguyên sau khi runtime bị xoá.

Siêu tham số phải **giống hệt** lần chạy đầu, nếu không trainer tính sai số
step còn lại. Muốn train thêm một epoch thì đổi `--epochs 1` thành `2`: đó là
học tiếp từ checkpoint, không phải train lại từ đầu.

In [ ]:
import os

from huggingface_hub import HfApi

DATA = f"{DRIVE_ROOT}/data/validated/sft"
OUTPUT_DIR = f"{DRIVE_ROOT}/outputs/sft"
HUB_MODEL_ID = f"{HfApi(token=os.environ['HF_TOKEN']).whoami()['name']}/chessvi-4b-sft"

!ls -la {OUTPUT_DIR} | head -20

!python -m chessvi.train.sft \
    --data {DATA} \
    --base-model Qwen/Qwen3-4B \
    --output-dir {OUTPUT_DIR} \
    --hub-model-id {HUB_MODEL_ID} \
    --epochs 1 --batch-size 8 --grad-accum 2 --save-steps 200 \
    --resume

## 10. Chốt chặn sau T8

Accuracy trên test set phải đạt **30–38%**. Dưới 20% nghĩa là dữ liệu có vấn
đề — quay lại T5, **đừng** chạy RL.

`--model-path` là **base model**, `--adapter` là thư mục LoRA vừa train.
`outputs/sft` chỉ chứa adapter chứ không chứa model, nên truyền nó vào
`--model-path` sẽ hỏng.

`--limit 300` để đọc nhanh: ở mức 33% thì 300 mẫu cho sai số khoảng ±5%, quá
đủ để phân biệt 20% với 33%. Bỏ `--limit` khi muốn số chính thức.

Prompt lúc eval là khuôn của T9 (`FEN: ... Nước đi nào?`), hơi khác khuôn C1
mà T8 vừa học (`FEN: ... Phân tích từng bước...`). `extract_move` có đường lui
lấy nước hợp lệ cuối cùng nên vẫn chấm đúng, nhưng con số này là **sàn** chứ
không phải trần — đúng khuôn thì sẽ cao hơn.

In [ ]:
PUZZLES = f"{DRIVE_ROOT}/data/puzzles/test.parquet"
REPORT = f"{DRIVE_ROOT}/reports/sft_acc.csv"

!mkdir -p {DRIVE_ROOT}/reports

!python -m chessvi.eval.puzzle_acc \
    --puzzles {PUZZLES} \
    --backend hf \
    --model-path Qwen/Qwen3-4B \
    --adapter {OUTPUT_DIR} \
    --model-name chessvi-4b-sft \
    --limit 300 \
    --out {REPORT}